In [114]:
!git clone https://github.com/mipaillafil/Proyecto_Movilidad_Cortes_Marchesse_Paillafil.git

fatal: destination path 'Proyecto_Movilidad_Cortes_Marchesse_Paillafil' already exists and is not an empty directory.


In [115]:
%cd /content/Proyecto_Movilidad_Cortes_Marchesse_Paillafil
!git pull


/content/Proyecto_Movilidad_Cortes_Marchesse_Paillafil
There is no tracking information for the current branch.
Please specify which branch you want to merge with.
See git-pull(1) for details.

    git pull <remote> <branch>

If you wish to set tracking information for this branch you can do so with:

    git branch --set-upstream-to=origin/<branch> limpieza-datos



In [117]:
!git branch

* limpieza-datos
  main


**CARGAR DATAFRAME**

In [96]:
import pandas as pd
import os
from google.colab import drive
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Definir Rutas (ACTUALIZADAS)
path_raiz = '/content/drive/MyDrive/'

path_raw = path_raiz + 'Quinto Semestre/Programación para la Ciencia de Datos/Evaluacion 1/Global_Mobility_Report.zip'

path_processed = path_raiz + 'Quinto Semestre/Programación para la Ciencia de Datos/Evaluacion 1/processed/'

# Crear carpeta processed si no existe
if not os.path.exists(path_processed):
    os.makedirs(path_processed)
    print(f"Carpeta creada en: {path_processed}")

# 3. Procesamiento por trozos
pedazos_chile = []

print("⏳ Iniciando lectura y filtrado...")

try:
    for chunk in pd.read_csv(path_raw, compression='zip', chunksize=100000, low_memory=False):
        filtrado = chunk[chunk['country_region'] == 'Chile']

        if not filtrado.empty:
            pedazos_chile.append(filtrado)

    # 4. Consolidación
    if pedazos_chile:
        df_chile = pd.concat(pedazos_chile, ignore_index=True)

        archivo_salida = os.path.join(path_processed, 'Chile_Mobility_Clean.csv')
        df_chile.to_csv(archivo_salida, index=False)

        print("✅ PROCESO COMPLETADO")
        print(f"📂 Archivo: {archivo_salida}")
        print(f"📊 Registros: {len(df_chile)}")
    else:
        print("⚠️ No hay datos de Chile")

except FileNotFoundError:
    print("❌ Ruta incorrecta, revisa nombres de carpetas")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⏳ Iniciando lectura y filtrado...
✅ PROCESO COMPLETADO
📂 Archivo: /content/drive/MyDrive/Quinto Semestre/Programación para la Ciencia de Datos/Evaluacion 1/processed/Chile_Mobility_Clean.csv
📊 Registros: 68716


In [97]:
df_chile.head(5)

,country_region_code,country_region,sub_region_1,sub_region_2,metro_area,iso_3166_2_code,census_fips_code,place_id,date,retail_and_recreation_percent_change_from_baseline,grocery_and_pharmacy_percent_change_from_baseline,parks_percent_change_from_baseline,transit_stations_percent_change_from_baseline,workplaces_percent_change_from_baseline,residential_percent_change_from_baseline
0,CL,Chile,NaN,NaN,NaN,NaN,NaN,ChIJL68lBEHFYpYRHbkCERPhBQU,2020-02-15,2.0,4.0,9.0,0.0,-3.0,0.0
1,CL,Chile,NaN,NaN,NaN,NaN,NaN,ChIJL68lBEHFYpYRHbkCERPhBQU,2020-02-16,3.0,5.0,5.0,4.0,-1.0,0.0
2,CL,Chile,NaN,NaN,NaN,NaN,NaN,ChIJL68lBEHFYpYRHbkCERPhBQU,2020-02-17,1.0,6.0,11.0,-3.0,-8.0,1.0
3,CL,Chile,NaN,NaN,NaN,NaN,NaN,ChIJL68lBEHFYpYRHbkCERPhBQU,2020-02-18,0.0,5.0,13.0,-3.0,-7.0,1.0
4,CL,Chile,NaN,NaN,NaN,NaN,NaN,ChIJL68lBEHFYpYRHbkCERPhBQU,2020-02-19,0.0,8.0,11.0,-3.0,-7.0,1.0


**LIMPIEZA**

In [98]:
#Verificar valores nulos
nulos = df_chile.isnull().sum()
print(f'NULOS POR COLUMNA:\n {nulos}')

print(f'\nPORCENTAJE DE NULOS POR COLUMNA:\n{(df_chile.isnull().mean() * 100).round(2)}')

NULOS POR COLUMNA:
 country_region_code                                       0
country_region                                            0
sub_region_1                                            974
sub_region_2                                          16556
metro_area                                            68716
iso_3166_2_code                                       53134
census_fips_code                                      68716
place_id                                                  0
date                                                      0
retail_and_recreation_percent_change_from_baseline     6295
grocery_and_pharmacy_percent_change_from_baseline     10520
parks_percent_change_from_baseline                      186
transit_stations_percent_change_from_baseline          8522
workplaces_percent_change_from_baseline                5585
residential_percent_change_from_baseline              10662
dtype: int64

PORCENTAJE DE NULOS POR COLUMNA:
country_region_code              

In [99]:
#Eliminar columnas con alto porcentaje de nulos (>20%)
columnas_eliminar = [
    'metro_area',            # 100%
    'census_fips_code',      # 100%
    'iso_3166_2_code',       # 77%
    'sub_region_2'           # 24%
]

df_chile.drop(columns=columnas_eliminar, inplace=True)

# Se rellenan valores en columnas con bajo porcentaje de nulos
df_chile['sub_region_1'] = df_chile['sub_region_1'].fillna('Sin dato')

In [100]:
#Ordenar por fecha
df_chile['date'] = pd.to_datetime(df_chile['date'])
df_chile = df_chile.sort_values('date')

In [101]:
columnas_numericas = [
    'retail_and_recreation_percent_change_from_baseline',
    'grocery_and_pharmacy_percent_change_from_baseline',
    'parks_percent_change_from_baseline',
    'transit_stations_percent_change_from_baseline',
    'workplaces_percent_change_from_baseline',
    'residential_percent_change_from_baseline'
]

# Forward fill
df_chile[columnas_numericas] = df_chile.groupby('sub_region_1')[columnas_numericas].ffill()

# Backward fill
df_chile[columnas_numericas] = df_chile.groupby('sub_region_1')[columnas_numericas].bfill()


In [102]:
print("\nTOTAL DE NULOS DESPUÉS DEL TRATAMIENTO:")
print(df_chile.isnull().sum())


TOTAL DE NULOS DESPUÉS DEL TRATAMIENTO:
country_region_code                                   0
country_region                                        0
sub_region_1                                          0
place_id                                              0
date                                                  0
retail_and_recreation_percent_change_from_baseline    0
grocery_and_pharmacy_percent_change_from_baseline     0
parks_percent_change_from_baseline                    0
transit_stations_percent_change_from_baseline         0
workplaces_percent_change_from_baseline               0
residential_percent_change_from_baseline              0
dtype: int64


In [103]:
#Verificar datos duplicados por fecha y región
duplicados_subregion = df_chile.duplicated(subset=['sub_region_1', 'date']).sum()
print(f'Duplicados por región completa: {duplicados_subregion}')

#Duplicados reales
duplicados_reales = df_chile.duplicated().sum()
print(f'Duplicados reales en el dataset: {duplicados_reales}')

Duplicados por región completa: 52160
Duplicados reales en el dataset: 0


In [104]:
for col in columnas_numericas:
    df_chile[col] = df_chile[col].clip(-100, 100)

In [105]:
print(df_chile.info())

<class 'pandas.core.frame.DataFrame'>
Index: 68716 entries, 0 to 68715
Data columns (total 11 columns):
 #   Column                                              Non-Null Count  Dtype         
---  ------                                              --------------  -----         
 0   country_region_code                                 68716 non-null  object        
 1   country_region                                      68716 non-null  object        
 2   sub_region_1                                        68716 non-null  object        
 3   place_id                                            68716 non-null  object        
 4   date                                                68716 non-null  datetime64[ns]
 5   retail_and_recreation_percent_change_from_baseline  68716 non-null  float64       
 6   grocery_and_pharmacy_percent_change_from_baseline   68716 non-null  float64       
 7   parks_percent_change_from_baseline                  68716 non-null  float64       
 8   transit_sta

**LIMPIEZA DE DATOS**

In [106]:
#Copia de df limpio
df_transformado = df_chile.copy()
df_transformado.head(5)

,country_region_code,country_region,sub_region_1,place_id,date,retail_and_recreation_percent_change_from_baseline,grocery_and_pharmacy_percent_change_from_baseline,parks_percent_change_from_baseline,transit_stations_percent_change_from_baseline,workplaces_percent_change_from_baseline,residential_percent_change_from_baseline
0,CL,Chile,Sin dato,ChIJL68lBEHFYpYRHbkCERPhBQU,2020-02-15,2.0,4.0,9.0,0.0,-3.0,0.0
21210,CL,Chile,Bio Bio,ChIJ1Z3A5N2kaZYRAEjv0m_GWO4,2020-02-15,-2.0,4.0,7.0,18.0,-3.0,-1.0
29975,CL,Chile,Los Lagos,ChIJPY-FNb_yHpYRYQiRBL6YFTk,2020-02-15,5.0,11.0,56.0,54.0,-1.0,1.0
41578,CL,Chile,Maule,ChIJoZflTaOWZZYR3ZTUCGqh6sI,2020-02-15,3.0,8.0,8.0,4.0,-4.0,-1.0
62872,CL,Chile,Valparaíso,ChIJYVcoefZ-YpYRxuUjLlagqUo,2020-02-15,3.0,1.0,8.0,-2.0,-3.0,0.0


In [107]:
#Convertir columna date a datetime
df_transformado['date'] = pd.to_datetime(df_transformado['date'])

In [108]:
# Crear columna de movilidad_promedio
df_transformado['movilidad_promedio'] = df_transformado[columnas_numericas].mean(axis=1)

In [109]:
#Escalamiento
scaler = StandardScaler()

df_transformado[columnas_numericas] = scaler.fit_transform(
    df_transformado[columnas_numericas]
)

In [111]:
#Codificación de la variable sub_region_1
df_transformado = pd.get_dummies(
    df_transformado,
    columns=['sub_region_1'],
    drop_first=True  # evita multicolinealidad
)

In [113]:
#Resultado final
print(f'DF LIMPIO Y TRANSFORMADO\n')
df_transformado.head(5)

DF LIMPIO Y TRANSFORMADO



,country_region_code,country_region,place_id,date,retail_and_recreation_percent_change_from_baseline,grocery_and_pharmacy_percent_change_from_baseline,parks_percent_change_from_baseline,transit_stations_percent_change_from_baseline,workplaces_percent_change_from_baseline,residential_percent_change_from_baseline,...,sub_region_1_Los Lagos,sub_region_1_Los Ríos,sub_region_1_Magallanes and Chilean Antarctica,sub_region_1_Maule,sub_region_1_O'Higgins,sub_region_1_Santiago Metropolitan Region,sub_region_1_Sin dato,sub_region_1_Tarapacá,sub_region_1_Valparaíso,sub_region_1_Ñuble
0,CL,Chile,ChIJL68lBEHFYpYRHbkCERPhBQU,2020-02-15,0.828457,0.222959,1.252432,0.576144,-0.010849,-1.791634,...,False,False,False,False,False,False,True,False,False,False
21210,CL,Chile,ChIJ1Z3A5N2kaZYRAEjv0m_GWO4,2020-02-15,0.696357,0.222959,1.188050,1.047941,-0.010849,-1.930575,...,False,False,False,False,False,False,False,False,False,False
29975,CL,Chile,ChIJPY-FNb_yHpYRYQiRBL6YFTk,2020-02-15,0.927532,0.448320,2.765408,1.991533,0.039480,-1.652692,...,True,False,False,False,False,False,False,False,False,False
41578,CL,Chile,ChIJoZflTaOWZZYR3ZTUCGqh6sI,2020-02-15,0.861482,0.351737,1.220241,0.680988,-0.036014,-1.930575,...,False,False,False,True,False,False,False,False,False,False
62872,CL,Chile,ChIJYVcoefZ-YpYRxuUjLlagqUo,2020-02-15,0.861482,0.126376,1.220241,0.523723,-0.010849,-1.791634,...,False,False,False,False,False,False,False,False,True,False


In [ ]:
from google.colab import _message
import json

# Esto descarga el contenido actual del notebook y lo guarda como un archivo .ipynb
with open('limpieza_datos_movilidad.ipynb', 'w') as f:
    f.write(json.dumps(_message.blocking_request('get_ipynb')['ipynb'], indent=1))

print("¡Archivo creado con éxito!")